In [ ]:
import numpy as np
import pyvista as pv
from pathlib import Path
import matplotlib.pyplot as plt

from qiskit_metal import MetalGUI, Dict
from SQDMetal.Utilities.Materials import MaterialInterface

from toolkit.tools.sim.save_load_design import save_design, load_design

from toolkit.tools.comp_manipulation.adjust_design import adjust_design
from toolkit.tools.sim import build_subdesign, run_eigenmode_sim, EigenmodeSimOptions, EigenmodePortOptions, MeshAlongPathOptions, AMROptions, FineMeshComponentOptions


In [ ]:
path2load = "circTransmon.json"

design = load_design(path2load)

# --- Meander resonator eigenmode simulation: run setup ---
path2palace = "/Users/maximilianogatto/Library/CloudStorage/OneDrive-Personal/PhD/src/libraries/palace/build/bin/palace"  # local PALACE binary

## Resonator

## Eigenfrequency

**What the solver does.** An eigenmode run has *no excitation*. It looks for the source-free
solutions of Maxwell's equations in the structure — the frequencies at which a field can sustain
itself. Dropping the source from the full-wave curl-curl equation leaves a generalised eigenvalue
problem for $(\omega,\mathbf E)$,

$$\nabla\times\left(\mu^{-1}\nabla\times\mathbf E\right)=\omega^2\varepsilon\,\mathbf E$$

which discretises to $K\mathbf e=\omega^2 M_\varepsilon\mathbf e$: the same curl-curl stiffness
matrix $K$ as in the magnetostatic section (see *Inductance* below for what $K$ is), now against
a mass matrix weighted by $\varepsilon$. SLEPc solves it by shift-and-invert around
`starting_freq`, which is why that option matters — it is where the search starts, not a filter.

**What comes back.** `data` is the raw `eig.csv` as an array, one row per mode, with columns

| index | column | units |
|---|---|---|
| 0 | mode number $m$ | — |
| 1 | $\mathrm{Re}\{f\}$ | **GHz** |
| 2 | $\mathrm{Im}\{f\}$ | **GHz** |
| 3 | $Q$ | — |
| 4 | error (backward) | — |
| 5 | error (absolute) | — |

Note columns 1 and 2 are in **GHz**, not Hz — everything else in this notebook (and every
`*SimOptions` frequency field) is in Hz, so the conversion has to happen here. And column 2 is the
imaginary part of the eigenfrequency (the linewidth), *not* the solver error; the error estimates
are columns 4 and 5.

**Where $Q$ comes from.** Not from the eigenvalue alone. The dielectric is lossless in the
eigenproblem unless you tell it otherwise, so $Q$ is reconstructed from the **energy participation
ratios**: the fraction of the mode's electric energy sitting in each lossy interface, weighted by
that interface's loss tangent. That is what `epr_interfaces` is for — omit it and you get
frequencies but no meaningful $Q$.

**What this particular run is.** The sub-design keeps only `meandro_1` + `stg1`: the **bare**
resonator, shorted at `stg1`, open at the (dropped) qubit end, with no feedline and no port. So
the frequency that comes out is the *unloaded* one. It will not match the dip in the driven $S_{21}$
below — the feedline and the qubit pull it. Here: 6.82 GHz bare against 6.688 GHz loaded.

In [ ]:
OUTPUT_EIGENMODE = "./resonator_eigensim"  # parent folder the run's output subfolder is created in

NUM_CPUS_EIGENMODE = 2  # MPI processes PALACE is launched with (-np)

In [ ]:
# Solver / mesh / eigenvalue-search settings for the bare meander resonator.
# This sub-design keeps only 'meandro_1' (the meander trace) + 'stg1' (its coupling stub) --
# no junction/port is kept, so PALACE just searches for the empty-cavity eigenmodes.
eigen_sim_options = EigenmodeSimOptions(
    mesh_refinement=0,          # global mesh refinement level (0 = coarsest/fastest)
    solns_to_save=1,            # number of eigenmode field solutions written to disk
    solver_order=1,             # finite-element order (1 or 2; 2 = more accurate, slower)
    solver_tol=1.0e-8,          # SLEPc eigenvalue solver tolerance
    solver_maxits=200,          # max solver iterations
    fillet_resolution=12,       # segments used to approximate a filleted corner
    palace_dir=path2palace,     # REQUIRED -- must be passed into run_eigenmode_sim below via
                                 # sim_options=, otherwise it silently falls back to "" and the
                                 # run fails with "-np: command not found"
    num_cpus=NUM_CPUS_EIGENMODE,
    meshing="GMSH",             # 'GMSH' (local, open-source) or 'COMSOL' (needs a license)
    mode="simPC",                # 'simPC' (run locally) or 'HPC' (write job files for a cluster)
    create_files=True,
    starting_freq=5e9,           # Hz -- SLEPc shift-invert search starts looking above this;
                                  # set near the meander's expected bare resonance
    number_of_freqs=1            # only searching for the fundamental mode (bump this up to
                                  # also catch higher harmonics, see the run's mode 2/3 output)
)

# Mesh refinement following the meander's centreline -- the field is concentrated right on the
# CPW trace, so this avoids either an over-coarse mesh there or an unnecessarily fine one
# everywhere else.
meander_mesh_options = MeshAlongPathOptions(
    qObjName="meandro_1",
    dist_resolution=10e-6,       # spacing between refinement points along the path, m
    min_size=6e-6,                # finest element size right on the trace, m
    max_size=150e-6,              # coarsest size the refinement tapers out to, m
    taper_dist_min=10e-6          # distance over which the taper starts, m
)

# Adaptive mesh refinement (AMR): PALACE re-meshes based on estimated solution error on top of
# the fixed fine-mesh spec above.
amr_options = AMROptions(
    num_iterations=5,             # max AMR passes
    max_DoFs=0,                   # cap on degrees of freedom (0 = no cap)
    tolerance=1e-2,                # stop refining once the error estimate drops below this
    Dorfler_marking_fraction=0.3,  # fraction of elements (by error) refined each pass
    save_iterations_data=False,
    save_iterations_mesh=False,
    nonconformal=True              # allow hanging nodes between refinement levels
)

# EPR (energy participation ratio) interfaces -- needed to get a lossy Q out of the sim (see the
# Q values in the run's output below), not just the bare frequency.
# run_eigenmode_sim unpacks this tuple as (substrate_air, substrate_metal, metal_air), so the
# order matters and it is NOT the alphabetical/obvious one:
#   substrate_air   = exposed silicon surface      -> 'Silicon-Vacuum'      (eps 3.85, tan 6e-3)
#   substrate_metal = silicon under the aluminium  -> 'Silicon-Aluminium'   (eps 10.0, tan 3e-4)
#   metal_air       = aluminium surface            -> 'Aluminium-Vacuum'    (eps 10.0, tan 5e-4)
epr_interfaces = (MaterialInterface('Silicon-Vacuum'),
                  MaterialInterface('Silicon-Aluminium'),
                  MaterialInterface('Aluminium-Vacuum'))

# --- Other options you could add here (left commented out) ---

# Extra mesh refinement grouped by component, in addition to the along-path one above -- useful
# e.g. around the coupling stub:
# from toolkit.tools.sim import FineMeshComponentOptions
# stg1_mesh_options = FineMeshComponentOptions(
#     qObjNames=["stg1"], min_size=8e-6, max_size=100e-6, taper_dist_min=10e-6, metals_only=False,
# )
# then pass fine_mesh_components=[stg1_mesh_options] into run_eigenmode_sim below.

# Surface kinetic inductance (Henry) -- models a lossy/kinetic-inductance metal (e.g. NbTiN)
# instead of the default purely geometric (lossless) one:
# kinetic_inductance = 2e-12
# then pass kinetic_inductance=kinetic_inductance into run_eigenmode_sim below.

# A port only applies if the sub-design keeps a junction or CPW feed component -- 'meandro_1' +
# 'stg1' alone have neither, so none is used here. If 'Q1' (with a junction) were added to
# `components` in the run cell below, you'd add:
# from toolkit.tools.sim import EigenmodePortOptions
# ports = [EigenmodePortOptions(component_name="Q1", type="JosephsonJunction", junction_index=0, L_J=10e-9, C_J=2e-15)]
# then pass ports=ports into run_eigenmode_sim below.

In [ ]:
# Run the eigenmode simulation for the bare meander resonator: keep only 'meandro_1' (the
# meander trace) and 'stg1' (its coupling stub), dropping everything else in `design`.
data = run_eigenmode_sim(design = design, 
                    name ="resonator_eigensim",                  # output subfolder name
                    output_path = OUTPUT_EIGENMODE, 
                    components = ['meandro_1', 'stg1'],           # only these two survive in the sub-design
                    open_terminations = [('meandro_1', 'end')],   # meandro_1.end used to connect to
                                                                   # Q1, which is dropped here -- so
                                                                   # terminate that dangling pin open
                    sim_options = eigen_sim_options,               # <-- must be passed explicitly, see
                                                                    # the comment on palace_dir above
                    amr_options = amr_options,
                    fine_mesh_paths = [meander_mesh_options],
                    epr_interfaces = epr_interfaces                # needed for Q, not just frequency
                    # ports=ports,                                  # uncomment if a junction/CPW port
                    #                                                # component is added to `components`
                    # kinetic_inductance=kinetic_inductance,         # uncomment for a lossy/kinetic metal
                    # terminations={('meandro_1', 'end'): 'short'},  # manual override instead of
                    #                                                 # open_terminations, e.g. to force
                    #                                                 # a short instead of an open
                    )

In [ ]:
# eig.csv columns: [m, Re{f} (GHz), Im{f} (GHz), Q, err_bkwd, err_abs]. Note GHz, and that
# column 2 is the imaginary part of the eigenfrequency (the linewidth), not the solver error --
# the error estimates are columns 4 and 5.
for point in data:
    print(f"Mode {int(point[0])}: {point[1]:.6f} GHz, Im(f) = {point[2]*1e3:.3g} MHz, "
          f"Q = {point[3]:.4g}, abs. error = {point[5]:.2e}")

# Every *SimOptions frequency field (center_freq, bandwidth, starting_freq, ...) is in Hz, so
# convert here rather than at each use site.
eigenfreq = data[0][1] * 1e9  # Hz

In [ ]:
from SQDMetal.PALACE.PVDVTU_DataReader import PVDVTU_DataReader
from SQDMetal.PALACE.PVDVTU_Viewer import PVDVTU_Viewer

In [ ]:
path2outfile = Path(OUTPUT_EIGENMODE) / 'resonator_eigensim' / "outputFiles" / "paraview" / "eigenmode" / "eigenmode.pvd"

reader = PVDVTU_DataReader(path2outfile) # to acces to the field and data

viewer = PVDVTU_Viewer(path2outfile) # to visualize the data in 3D
viewer.num_datasets

In [ ]:
leSlice = viewer.get_data_slice(data_set_index=0,slice_plane_normal=[0,0,1], slice_plane_origin=[0,0,0])

print("Available parameters for the slice:")
print(leSlice.get_params())

In [ ]:
E_real = leSlice.get_data(param_name='E_real')

print(E_real)

In [ ]:
# E_real is a vector field (Ex, Ey, Ez per point) -- plot() needs a scalar array per point
# (see its docstring), so reduce it to the field magnitude first. Swap in e.g. E_real[:, 2] to
# plot a single component (Ez) instead, if that's what you want to look at.
E_mag = np.linalg.norm(E_real, axis=1)

leSlice.plot(E_mag, cmap='magma', equalise_histogram=True)

plt.show()

## Driven

**What the solver does.** Unlike the eigenmode run, this one is a *full-wave* problem with a
source: at each frequency it solves

$$\left[\nabla\times\left(\mu^{-1}\nabla\times\right)-\omega^2\varepsilon\right]\mathbf E=-i\omega\mathbf J$$

with lumped ports on the launchpads, and reports the scattering parameters. Both time-derivative
terms of Maxwell are kept, so unlike the electrostatic and magnetostatic runs further down this
one contains **capacitive and inductive coupling simultaneously, including their interference and
retardation**. That makes the driven sweep the ground truth against which the $C$ and $M$ matrices
extracted separately below should be checked, not the other way round.

**Ports.** One `DrivenModalPortOptions` per launchpad; `excite=True` marks the driving port. With
`LP1` excited and `LP2` passive, `driven_result['S21']` is transmission through the feedline, and
the resonator shows up as a dip where it loads the line.

**Reading the dip.** The 3 dB width gives the loaded $Q_L=f_r/\Delta f$, which combines the
internal and external contributions as $Q_L^{-1}=Q_i^{-1}+Q_e^{-1}$. The width estimator used
below is crude — it walks outwards until the numerical derivative falls under a threshold, so it
is sensitive to `freq_points` and to noise on the sweep. For a number you intend to quote, fit a
Lorentzian (or better, a circle fit in the complex plane, which also separates $Q_i$ from $Q_e$
and absorbs the line asymmetry) rather than reading the width off the magnitude.

In [ ]:
from toolkit.tools.sim import run_drivenmodal_sim, DrivenModalSimOptions, DrivenModalPortOptions

Palace has two modes for frequency-domain driven simulations: uniform and adaptive. The uniform solver runs one full finite-element solve per output frequency. The adaptive solver runs a much smaller number of full solves, constructs a projection-based reduced-order model (PROM), and evaluates that model over the requested output grid.

It is much faster that solve full system in each point, but can fails with high-Q resonances. In that case, the uniform solver is more robust.

Ansys uses a similar approach, called "adaptive frequency sweeps" (AFS). For that reason, you can find Asys "faster" than Palace full sweep, but it is due to the fact that Ansys uses a reduced-order model. To compare both, you can use the adaptive solver in Palace, or full solution in Ansys.

source: https://awslabs.github.io/palace/stable/features/adaptive_driven_solver/#Adaptive-Frequency-Sweeps-for-Driven-Simulations

In [ ]:
f_center = eigenfreq
f_bandwidth = 1e9
N_freq_points = 1001

OUTPUT_DRIVEN = "./resonator_drivensim"  # parent folder the run's output subfolder is created in

NUM_CPUS_DRIVEN = 4 # MPI processes PALACE is launched with (-np)

driven_options = DrivenModalSimOptions(
    mesh_refinement=0,
    solns_to_save=N_freq_points,
    solver_order=1,
    solver_tol=1.0e-8,
    adaptive_solver_tol=1e5,
    palace_dir=path2palace,
    num_cpus=NUM_CPUS_DRIVEN,
    center_freq=f_center,
    bandwidth=f_bandwidth,
    freq_points=N_freq_points
)

ports = [DrivenModalPortOptions(component_name="LP1", excite=True), DrivenModalPortOptions(component_name="LP2", excite=False)]

meander_mesh_options = MeshAlongPathOptions(
    qObjName="meandro_1",
    dist_resolution=10e-6,       # spacing between refinement points along the path, m
    min_size=6e-6,                # finest element size right on the trace, m
    max_size=150e-6,              # coarsest size the refinement tapers out to, m
    taper_dist_min=10e-6          # distance over which the taper starts, m
)

feedline_mesh_options = MeshAlongPathOptions(
    qObjName="linea_1",
    dist_resolution=10e-6,       # spacing between refinement points along the path, m
    min_size=6e-6,                # finest element size right on the trace, m
    max_size=150e-6,              # coarsest size the refinement tapers out to, m
    taper_dist_min=10e-6          # distance over which the taper starts, m
)

In [ ]:
driven_result = run_drivenmodal_sim(design = design,
                    name ="resonator_drivensim",                  # output subfolder name
                    output_path = OUTPUT_DRIVEN,
                    components = ['meandro_1', 'stg1', 'linea_1', 'LP1', 'LP2'],
                    open_terminations = [('meandro_1', 'end')],
                    sim_options = driven_options,
                    ports = ports,
                    fine_mesh_paths = [meander_mesh_options, feedline_mesh_options]
                    )

In [ ]:
print(driven_result.keys())

freq = driven_result['freqs']
s21 = driven_result['S21']

In [ ]:
mag_s21 = 20 * np.log10(np.abs(s21))
phase = np.angle(s21, deg=True)

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 8))
ax[0].plot(freq, mag_s21)
ax[0].set_ylabel('Magnitude [dB]')
ax[1].plot(freq, phase)
ax[1].set_ylabel('Phase [degrees]')
ax[1].set_xlabel('Frequency [Hz]')

index_min = np.argmin(mag_s21)
f_res = freq[index_min]
# estimation bw
dif = np.diff(mag_s21)
epsilon = 1e-3

index_bw = np.where(abs(dif) < epsilon)[0]
index_3dB_low = index_bw[index_bw < index_min][-1]
index_3dB_high = index_bw[index_bw > index_min][0]
f_3dB_low = freq[index_3dB_low]
f_3dB_high = freq[index_3dB_high]
bw = f_3dB_high - f_3dB_low


ax[0].axvline(f_res, color='r', linestyle='--', label='Resonance Frequency')
ax[0].axvline(f_3dB_low, color='g', linestyle='--', label='3 dB Low')
ax[0].axvline(f_3dB_high, color='g', linestyle='--', label='3 dB High')
ax[0].legend()

print(f"Minimum S21: {mag_s21[index_min]:.2f} dB at {freq[index_min]/1e9:.3f} GHz")
print(f"3 dB bandwidth: {bw/1e6:.2f} MHz")
print(f"3 dB low: {f_3dB_low/1e9:.3f} GHz")
print(f"3 dB high: {f_3dB_high/1e9:.3f} GHz") 

In [ ]:
# zoom on res_freq plot
plot_bw = 10e6
index = np.where((freq > f_res - plot_bw/2) & (freq < f_res + plot_bw/2))

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 8))
ax[0].plot(freq[index], mag_s21[index])
ax[0].set_ylabel('Magnitude [dB]')
ax[1].plot(freq[index], phase[index])
ax[1].set_ylabel('Phase [degrees]')
ax[1].set_xlabel('Frequency [Hz]')



In [ ]:
f_center = f_res
f_bandwidth = bw * 3
N_freq_points = 4001

driven_options.center_freq = f_center
driven_options.bandwidth = f_bandwidth
driven_options.freq_points = N_freq_points

In [ ]:
driven_result_fine = run_drivenmodal_sim(design = design,
                    name ="resonator_drivensim",                  # output subfolder name
                    output_path = OUTPUT_DRIVEN,
                    components = ['meandro_1', 'stg1', 'linea_1', 'LP1', 'LP2'],
                    open_terminations = [('meandro_1', 'end')],
                    sim_options = driven_options,
                    ports = ports,
                    fine_mesh_paths = [meander_mesh_options, feedline_mesh_options]
                    )

In [ ]:
freq = driven_result_fine['freqs']
s21 = driven_result_fine['S21']

In [ ]:
mag_s21 = 20 * np.log10(np.abs(s21))
phase = np.angle(s21, deg=True)

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 8))
ax[0].plot(freq, mag_s21)
ax[0].set_ylabel('Magnitude [dB]')
ax[1].plot(freq, phase)
ax[1].set_ylabel('Phase [degrees]')
ax[1].set_xlabel('Frequency [Hz]')

index_min = np.argmin(mag_s21)
f_res = freq[index_min]
# estimation bw
dif = np.diff(mag_s21)
epsilon = 1e-3

index_bw = np.where(abs(dif) < epsilon)[0]
index_3dB_low = index_bw[index_bw < index_min][-1]
index_3dB_high = index_bw[index_bw > index_min][0]
f_3dB_low = freq[index_3dB_low]
f_3dB_high = freq[index_3dB_high]
bw = f_3dB_high - f_3dB_low


ax[0].axvline(f_res, color='r', linestyle='--', label='Resonance Frequency')
ax[0].axvline(f_3dB_low, color='g', linestyle='--', label='3 dB Low')
ax[0].axvline(f_3dB_high, color='g', linestyle='--', label='3 dB High')
ax[0].legend()

print(f"Minimum S21: {mag_s21[index_min]:.2f} dB at {freq[index_min]/1e9:.3f} GHz")
print(f"3 dB bandwidth: {bw/1e6:.2f} MHz")
print(f"3 dB low: {f_3dB_low/1e9:.3f} GHz")
print(f"3 dB high: {f_3dB_high/1e9:.3f} GHz") 

## Capacitance

**What the solver does.** The electrostatic limit is the complementary one to the magnetostatic
run at the bottom of this notebook: there we dropped $\partial\mathbf D/\partial t$, here we drop
$\partial\mathbf B/\partial t$ and are left with

$$\nabla\cdot(\varepsilon\nabla\varphi)=0,\qquad \mathbf E=-\nabla\varphi$$

Each conductor is a `Terminal` boundary held at a fixed potential; PALACE solves once per
terminal ($V_i=1$, all others **grounded**) and reads the capacitance matrix off the electric
energy exactly as the magnetostatic solver reads $M$ off the magnetic energy,
$C_{ij}=\partial^2U_e/\partial V_i\partial V_j$.

**The duality, and where it inverts.** Worth keeping straight, because the two runs need opposite
geometry:

| | Capacitance (electrostatic) | Inductance (magnetostatic) |
|---|---|---|
| index $i$ is | a conductor **body** | a **cut** (terminal patch) |
| conductor must be | galvanically **isolated** | galvanically **closed to ground** |
| excitation | fix $V_i=1$ | inject $I_i=1$ A at the cut |
| the others are | **grounded** ($V_j=0$) | **open** ($I_j=0$) |

That is why the meander is terminated `'open'` at *both* ends here (`open_terminations` on
`start` and `end`): an isolated island is what makes it a well-defined node. Doing the same thing
in the inductance run is what makes that problem unsolvable — see the *Inductance* section.

**Maxwell vs mutual matrix — read this before using a number.** `run_capacitance_sim` returns
PALACE's `terminal-C.csv`, the **Maxwell** capacitance matrix: $Q_i=\sum_j C_{ij}V_j$, with
positive diagonal and negative off-diagonal. The physical (SPICE) capacitance between conductors
$i$ and $j$ is $-C_{ij}$, so the `abs()` below is not cosmetic. The diagonal is that conductor's
total capacitance to everything else, and a row that does not sum to zero is capacitance escaping
to the far-field boundary.

Careful: the analogous convention does **not** apply to inductance. `terminal-M.csv` is also the
"Maxwell" matrix, but there the off-diagonal *is* the physical mutual inductance
($\Phi_1=L_1I_1+MI_2$) with no sign flip; the negated version is the separate `terminal-Mm.csv`,
which SQDMetal does not read. So: negate for $C$, do not negate for $M$.

In [ ]:
from toolkit.tools.sim import run_capacitance_sim, CapacitanceSimOptions

### Capacitance of coupling resonator and feedline

In [ ]:
OUTPUT_CAPACITANCE = "./resonator_capacitance"  # parent folder the run's output subfolder is created in
NUM_CPUS_CAPACITANCE = 4  # MPI processes PALACE is launched with (-np)

cap_options = CapacitanceSimOptions(
    solver_order=2,
    palace_dir=path2palace,
    num_cpus=NUM_CPUS_CAPACITANCE)

meander_mesh_options = MeshAlongPathOptions(
    qObjName="meandro_1",
    dist_resolution=10e-6,       # spacing between refinement points along the path, m
    min_size=6e-6,                # finest element size right on the trace, m
    max_size=150e-6,              # coarsest size the refinement tapers out to, m
    taper_dist_min=10e-6          # distance over which the taper starts, m
)

feedline_mesh_options = MeshAlongPathOptions(
    qObjName="linea_1",
    dist_resolution=10e-6,       # spacing between refinement points along the path, m
    min_size=6e-6,                # finest element size right on the trace, m
    max_size=150e-6,              # coarsest size the refinement tapers out to, m
    taper_dist_min=10e-6          # distance over which the taper starts, m
)

In [ ]:
resonator_cap_res = run_capacitance_sim(design = design,
                    name ="resonator_capacitance",                  # output subfolder name
                    output_path = OUTPUT_CAPACITANCE,
                    components = ['meandro_1', 'linea_1', 'LP1', 'LP2'],
                    open_terminations = [('meandro_1', 'end'), ('meandro_1', 'start')],
                    sim_options = cap_options,
                    fine_mesh_paths = [meander_mesh_options, feedline_mesh_options]
                    )



In [ ]:
resonator_cap_res

In [ ]:
cupling_cap = abs(resonator_cap_res['Cond3']["launch_pad_LP1"])

print(f"Coupling capacitance: {cupling_cap:.2f} fF")

### Qubit capacitance

In [ ]:
OUTPUT_CAPACITANCE_QUBIT = "./qubit_capacitance"  # parent folder the run's output subfolder is created in

qubit_mesh = FineMeshComponentOptions(['Q1'], min_size=6e-6, max_size=100e-6, taper_dist_min=10e-6, metals_only=False)

In [ ]:
qubit_cap_res = run_capacitance_sim(design = design,
                    name ="qubit_capacitance",                  # output subfolder name
                    output_path = OUTPUT_CAPACITANCE_QUBIT,
                    components = ['Q1'],
                    open_terminations = [('Q1', 'claw_a'), ('Q1', 'charge_line')],
                    sim_options = cap_options,
                    fine_mesh_components = [qubit_mesh]          # meander/feedline mesh options dropped --
                                                                  # neither component exists in this
                                                                  # ['Q1']-only sub-design
                    )

In [ ]:
qubit_cap_res

In [ ]:
Cq = abs(qubit_cap_res['ground_main_plane']["pads_Q1"])
Ccl = abs(qubit_cap_res['charge_line_Q1']["pads_Q1"])
Cg = abs(qubit_cap_res['claw_a_Q1']["pads_Q1"]) 

print(f"Qubit capacitance to ground: {Cq:.2f} fF")
print(f"Qubit capacitance to claw: {Cg:.2f} fF")
print(f"Qubit capacitance to charge line: {Ccl:.2f} fF")

## Inductance

### From Maxwell's equations to a magnetostatic BVP

Start from the full set,

$$\nabla\times\mathbf E=-\frac{\partial\mathbf B}{\partial t},\qquad
\nabla\times\mathbf H=\mathbf J+\frac{\partial\mathbf D}{\partial t},\qquad
\nabla\cdot\mathbf B=0,\qquad \nabla\cdot\mathbf D=\rho$$

and take the **magnetoquasistatic (MQS)** limit: the circuit is small compared with a
wavelength. Here the coupler is ~460 µm and $\lambda$ at 6.8 GHz in silicon is ~18 mm, so
$\ell/\lambda\sim0.03$. In that limit the **displacement current $\partial\mathbf D/\partial t$
is dropped**, leaving

$$\nabla\times\mathbf H=\mathbf J,\qquad \nabla\cdot\mathbf B=0$$

**Faraday's law is *not* dropped.** That is the whole trick, and it is why a static solve gives
you a quantity that lives in a time-dependent circuit equation. The hierarchy is:

1. $\mathbf B$ follows from $\mathbf J$ through Ampère, *instantaneously* — no retardation, no
   waves. This is a **static** problem even if $\mathbf J(t)$ varies in time.
2. $\mathbf E$ follows afterwards from Faraday, driven by $\partial\mathbf B/\partial t$.

PALACE does step 1. Time re-enters in step 2, which you do by hand — and that is where
$V=L\,dI/dt$ comes from (§ *Why this is the $L$ in $V=L\,dI/dt$* below).

Since $\nabla\cdot\mathbf B=0$ there is a vector potential, $\mathbf B=\nabla\times\mathbf A$;
substituting $\mathbf H=\mu^{-1}\mathbf B$ into Ampère gives what the solver actually discretises:

$$\nabla\times\left(\mu^{-1}\nabla\times\mathbf A\right)=\mathbf J$$

Expanding the double curl with the vector identity $\nabla\times(\nabla\times\mathbf A)=
\nabla(\nabla\cdot\mathbf A)-\nabla^2\mathbf A$ — the analogue of the BAC-CAB rule, and note it is
an identity in its own right, not a consequence of BAC-CAB, since $\nabla$ does not commute with
what it acts on — and using that $\mu$ is uniform here ($\mu_r=1$ in both air and silicon; see the
`Domains` block of the generated config), the equation becomes

$$\nabla(\nabla\cdot\mathbf A)-\nabla^2\mathbf A=\mu\mathbf J$$

and in the Coulomb gauge $\nabla\cdot\mathbf A=0$, a vector Poisson equation:

$$\nabla^2\mathbf A=-\mu\mathbf J$$

(Careful: $\nabla^2\mathbf A$ is the componentwise scalar Laplacian *only in Cartesian
coordinates*, where the basis vectors are constant. The coordinate-free definition of the vector
Laplacian is the identity above.)

That form is worth writing down because it is the **analytic** route to the same answer. Its
free-space Green's function gives $\mathbf A=\frac{\mu}{4\pi}\int\mathbf J'/|\mathbf r-\mathbf
r'|\,dV'$, i.e. Biot–Savart; feeding that into the energy $U_m=\frac12\int\mathbf A\cdot\mathbf
J\,dV$ and reducing the currents to filaments, $\mathbf J\,dV\to I\,d\boldsymbol\ell$, gives

$$U_m=\frac{\mu}{8\pi}\sum_{ij}I_iI_j\oint_{C_i}\oint_{C_j}
\frac{d\boldsymbol\ell_i\cdot d\boldsymbol\ell_j}{|\mathbf r_i-\mathbf r_j|}
\;\equiv\;\frac12\sum_{ij}M_{ij}I_iI_j
\quad\Longrightarrow\quad
M_{ij}=\frac{\mu_0}{4\pi}\oint_{C_i}\oint_{C_j}\frac{d\boldsymbol\ell_i\cdot d\boldsymbol\ell_j}{|\mathbf r_i-\mathbf r_j|}$$

the Neumann formula — the same $M$ that comes out of `terminal-M.csv`. (Its diagonal diverges
logarithmically for a zero-radius filament: a self-inductance needs a finite conductor
cross-section, whereas the mutual is finite for any two disjoint loops. That asymmetry survives
into the numerics, and is why the diagonal is the sensitive number and the off-diagonal the robust
one.)

**PALACE does not discretise the Poisson form, though.** Three reasons, and the first is the next
paragraph:

In the meshed volume (air + dielectric) there is **no current**: $\mathbf J=0$ there, all of it
lives on the metal surfaces. So in the volume the equation is homogeneous, $=0$, and the current
enters entirely through the boundary conditions:

* **PEC** (`Boundaries.PEC`) on every metal and on the far-field box. Metals are *zero-thickness
  perfect sheets*: flux does not penetrate, $\mathbf B\cdot\hat n=0$, which in terms of $\mathbf
  A$ is $\hat n\times\mathbf A=0$ (a Dirichlet condition). The sheet carries whatever surface
  current is needed to enforce that; that current is never solved for as an unknown, and there
  is no penetration depth, no kinetic inductance, no normal-metal resistance. PALACE prints this
  itself: *"Magnetostatic problem type does not account for material permittivity, loss tangent,
  electrical conductivity, or London depth"*.
* **SurfaceCurrent** terminals (`Boundaries.SurfaceCurrent`) — one per
  `create_current_source_with_Uclip_on_*` call. A patch on which a surface current density
  $\mathbf K$ is *imposed*, i.e. a jump $\hat n\times[\mathbf H]=\mathbf K$. It is a Neumann
  condition: it enters the right-hand side of the linear system as $\int_\Gamma\mathbf
  K\cdot\mathbf A'\,dS$. In `palace/models/surfacecurrentoperator.cpp`, `Jinc = 1.0 /
  (width * n_elems)` — a total of 1 A spread uniformly over the width of the patch.

That is the entire problem. No time derivative anywhere in it.

### Where the inductance matrix comes from

There are two equivalent readings of the same field solution.

**Via flux.** By Stokes, the flux linked by loop $i$ is the circulation of $\mathbf A$,
$\Phi_i=\oint_{C_i}\mathbf A\cdot d\boldsymbol\ell$. The problem is linear in $\mathbf J$, so
$\Phi_i=\sum_j M_{ij}I_j$ — which *is* the definition of $M$: flux in loop $i$ per unit current
in loop $j$.

**Via energy.** Integrating by parts and using $\nabla\times\mathbf H=\mathbf J$,

$$U_m=\frac12\int\mathbf B\cdot\mathbf H\,dV=\frac12\int\mathbf A\cdot\mathbf J\,dV
=\frac12\sum_i I_i\oint_{C_i}\mathbf A\cdot d\boldsymbol\ell=\frac12\sum_{ij}M_{ij}I_iI_j
\;\;\Longrightarrow\;\; M_{ij}=\frac{\partial^2U_m}{\partial I_i\,\partial I_j}$$

Same $M$. PALACE solves once per terminal (terminal $i$ excited, the rest at zero), getting
$\mathbf A_i$, and reads the matrix off the second form. From
`palace/drivers/magnetostaticsolver.cpp::PostprocessTerminals`:

```
// Diagonal:     Mii = 2 Um(Ai) / Ii^2            = (Ai^T K Ai) / Ii^2
// Off-diagonal: Mij = Um(Ai+Aj)/(Ii Ij) - ...    = (Aj^T K Ai) / (Ii Ij)
```

$K$ is the curl-curl stiffness matrix, so $\mathbf A_j^TK\mathbf A_i=\int\mu^{-1}
\mathbf B_j\!\cdot\!\mathbf B_i\,dV$: the off-diagonal is the **cross energy** of the two
single-terminal solutions, i.e. the polarisation identity applied to $U_m$. The energy form is
preferred because it is a volume inner product; the flux form would force you to pick a surface
and integrate $\nabla\times\mathbf A$ locally, which is numerically much worse. That trade-off
is the comment in that same function.

The result lands in `terminal-M.csv`, in henries. (`terminal-Mm.csv` is the *mesh/mutual*
convention $M_m$, the magnetic analogue of the SPICE-vs-Maxwell capacitance distinction — the
number you want for $\Phi_1=L_1I_1+MI_2$ is the off-diagonal of `terminal-M.csv`.)


### What $K$ actually is: the curl-curl matrix

$K$ appears in the formulas above without explanation, and it is worth unpacking because it is
easy to confuse with $M$ — they are entirely different objects.

The field $\mathbf A(\mathbf r)$ has infinitely many degrees of freedom. The finite-element
method replaces it by a finite sum over **known, fixed** basis functions:

$$\mathbf A(\mathbf r)\;\approx\;\sum_{n=1}^{N}a_n\,\mathbf w_n(\mathbf r)$$

The $\mathbf w_n$ are Nédélec (edge) functions — one per mesh edge at lowest order, more per edge
and face at `solver_order=2`. Their defining property is that the coefficient $a_n$ is the
**circulation of $\mathbf A$ along that edge**, $\int\mathbf A\cdot d\boldsymbol\ell$ — exactly
the quantity that appears in $\Phi=\oint\mathbf A\cdot d\boldsymbol\ell$. For this run
$N=887816$ (the log line `ND (p = 2): 887816`). So `A[step]` in PALACE's source is *not* the
field: it is the vector of $N$ coefficients.

Substituting that expansion into the energy turns a functional into a quadratic form:

$$2U_m=\int\mu^{-1}|\nabla\times\mathbf A|^2dV
=\sum_{n,m}a_na_m\underbrace{\int\mu^{-1}(\nabla\times\mathbf w_n)\cdot(\nabla\times\mathbf w_m)\,dV}_{\textstyle K_{nm}}
=\mathbf a^TK\,\mathbf a$$

so

$$K_{nm}=\int\mu^{-1}\,(\nabla\times\mathbf w_n)\cdot(\nabla\times\mathbf w_m)\,dV$$

A curl on each side of the integrand — hence *curl-curl*. It is the discrete form of the operator
$\nabla\times(\mu^{-1}\nabla\times\,\cdot\,)$, it is $N\times N$ but **sparse** ($K_{nm}\neq0$
only when edges $n$ and $m$ share a tetrahedron), symmetric positive semi-definite, and it
depends on **the mesh and $\mu$ alone** — nothing about terminals or excitation. That is why
PALACE assembles it once, before the loop over terminals:
`auto K = curlcurl_op.GetStiffnessMatrix();`

**Where the current enters.** Not into $K$, but into the right-hand side:

$$K\,\mathbf a_i=\mathbf b_i,\qquad (\mathbf b_i)_n=\int_{\Gamma_i}\mathbf K_{\rm surf}\cdot\mathbf w_n\,dS$$

with $\Gamma_i$ the terminal patch. $(\mathbf b_i)_n$ is non-zero **only for basis functions whose
support touches that patch** — a few hundred non-zero entries in a vector of ~900 000, all
concentrated on a 20 × 20 µm square. Solving $K\mathbf a=\mathbf b$ is what spreads that into a
field over the whole chip. Exciting one terminal at a time means building a different
$\mathbf b_i$ each pass and reusing the same $K$: that is the two lines
`GetExcitationVector(idx, RHS)` then `ksp.Mult(RHS, A[step])`. A terminal that is "off" simply
contributes nothing to $\mathbf b$ — not a constraint, an absence, which is why it behaves as an
open circuit.

**Two matrices, wildly different.**

| | $K$ (curl-curl) | $M$ (inductance) |
|---|---|---|
| size | $N\times N$ = 887816 × 887816 | $n\times n$ = 2 × 2 |
| indices are | mesh degrees of freedom | terminals |
| depends on | mesh + $\mu$ | where you put the cuts |
| role | the operator being inverted | the answer |

$M$ is not $K$. $K$ is the machinery; $M$ is a tiny matrix distilled from it by sandwiching $K$
between the solution vectors — and the *only* difference between a self- and a mutual inductance
is **which pair of solutions you plug in**:

$$M_{11}=\frac{\mathbf a_1^TK\mathbf a_1}{I_1^2},\qquad
M_{22}=\frac{\mathbf a_2^TK\mathbf a_2}{I_2^2},\qquad
M_{12}=\frac{\mathbf a_1^TK\mathbf a_2}{I_1I_2}$$

the same vector twice → self; two different vectors → mutual. $K$ knows nothing about which metal
is the feedline; $\mathbf a_1$ does, because it is the field pattern produced when current
circulates in *that* loop.

**Why this is still the flux definition.** Since $\mathbf a_j=K^{-1}\mathbf b_j$, the $K$ cancels:

$$M_{ij}=\frac{\mathbf a_i^TK\,\mathbf a_j}{I_iI_j}=\frac{\mathbf a_i^T\mathbf b_j}{I_iI_j}$$

and because $\mathbf b_j$ lives only on patch $j$, that sum runs over the degrees of freedom of
that patch alone. On PEC the boundary condition is $\hat n\times\mathbf A=0$, i.e. the tangential
$\mathbf A$ vanishes on *all* the metal — so going around loop $j$, the stretches along the trace
and the ground plane contribute nothing to $\oint\mathbf A\cdot d\boldsymbol\ell$ and the entire
circulation is concentrated at the cut, which is precisely where $\mathbf b_j$ lives. Hence
$\mathbf a_i^T\mathbf b_j=I_j\Phi_j^{(i)}$ and

$$M_{ij}=\frac{\Phi_j^{(i)}}{I_i}$$

the flux of loop $i$'s field through loop $j$ per unit current. The two forms are algebraically
identical; PALACE uses the energy one because, as its own comment says, it avoids integrating
$\mathbf B=\nabla\times\mathbf A$ locally over a surface — and a volume integral over the whole
domain is far more forgiving than a local one right at the patch, where the mesh is worst and the
field has edge singularities.

**An inner product, and what it buys.** $K$ defines $\langle\mathbf x,\mathbf
y\rangle_K=\int\mu^{-1}\mathbf B_x\cdot\mathbf B_y\,dV$. Then $M_{ii}=\|\mathbf B_i\|^2/I_i^2$ is
a squared norm — always positive, "how much field energy it costs to push 1 A around that loop" —
while $M_{ij}$ is the **overlap** of two field patterns, and its sign depends on the `Direction`
vectors chosen for the two terminals in the config, which is a convention (hence the `abs()`
below). And Cauchy–Schwarz in that inner product,

$$|\langle\mathbf B_1,\mathbf B_2\rangle|\le\|\mathbf B_1\|\|\mathbf B_2\|
\quad\Longleftrightarrow\quad|M_{12}|\le\sqrt{M_{11}M_{22}}\quad\Longleftrightarrow\quad k\le1,$$

is exactly what `k_coup` checks in the cell below. It is not a rule of thumb from circuit
theory: $k>1$ means what came back is not a field, and no amount of post-processing will fix it.

### Why this is the $L$ in $V = L\,dI/dt$

Now apply Faraday to loop $i$: $\oint_{C_i}\mathbf E\cdot d\boldsymbol\ell=-d\Phi_i/dt$. The loop
runs inside perfect metal, where $\mathbf E=0$, **except across the cut** — the terminal patch.
So the whole circulation is concentrated there, and the voltage across the cut is

$$V_i=\frac{d\Phi_i}{dt}=\sum_j M_{ij}\frac{dI_j}{dt}
\qquad\Longrightarrow\qquad
V_1=L_1\dot I_1+M\dot I_2,\qquad V_2=M\dot I_1+L_2\dot I_2$$

So the diagonal of `terminal-M.csv` is exactly the $L$ of $V=L\,dI/dt$, and the off-diagonal is
the $M$ of "the EMF induced in the resonator by $dI_\text{feedline}/dt$" — the magnetic coupling
we are after. The solver never needs to do anything dynamic because in MQS $M$ is **purely
geometric**: the same number at DC and at 6.8 GHz. What changes with frequency is how much
current flows, not how much flux per ampere.

### The rule that decides whether your setup is valid

Take the divergence of Ampère without the displacement term:

$$\nabla\cdot(\nabla\times\mathbf H)=0=\nabla\cdot\mathbf J$$

**Ampère without $\partial\mathbf D/\partial t$ only has a solution for solenoidal $\mathbf J$.**
Current cannot pile up anywhere, because the term that would allow it to — $\partial\rho/\partial
t$, through $\partial\mathbf D/\partial t$ — is precisely the one that was dropped.

> **A current terminal must be a cut in a galvanically CLOSED loop:** both sides of the terminal
> patch have to belong to the *same* connected metal, so the current pushed in at the cut can
> flow back around and return to it.

The algebraic face of the same statement: $K$ has a null space, the gradients $\nabla\varphi$ of
any potential constant on each connected PEC body, so a solution exists only if the right-hand
side is orthogonal to it — i.e. only if the net current injected into every connected piece of
metal is zero. That orthogonality condition *is* $\nabla\cdot\mathbf J=0$ in weak form.

A conductor that only touches ground through the terminal itself is a dead end. The system is
inconsistent, CG stalls or diverges — while PALACE still writes a full `terminal-M.csv` of
garbage, which is why `run_inductance_sim` now checks the log for convergence by default.

### Why "isolate the resonator and watch the current get induced" cannot work

Two independent reasons. First, there is no $\partial\mathbf B/\partial t$ in the equation being
solved: the solver **cannot produce an induced current**, only the static field of currents you
impose. Second, even as a definition, a floating conductor has no loop for DC to circulate in.

This is the exact dual of the capacitance simulation, and the duality inverts the recipe:

| | Capacitance (electrostatic) | Inductance (magnetostatic) |
|---|---|---|
| unknowns attach to | **nodes** (isolated islands) | **loops** (closed circuits) |
| conductor must be | galvanically **isolated** — that is what makes it a node | galvanically **closed to ground** — that is what makes it a loop |
| excitation | fix a potential | inject a current at a cut |
| floating conductor | well defined ($Q$ on a floating island is static) | **undefined** — DC cannot circulate |

So the `OpenToGround`-at-both-ends trick that was right for $C$ is exactly what makes the
magnetostatic problem unsolvable. Each conductor instead needs a **short to ground at one end
and the terminal at the other**:

* **resonator** — already shorted by `stg1`, so the terminal goes at the free end. The loop is
  `meandro_1.end` → U-clip → ground plane → around → `stg1` → up the meander → back to the
  terminal. That loop is the resonator's real current path (a CPW returns its current in the
  adjacent ground planes), so $M_{12}$ from it is the physically meaningful coupling inductance.
* **feedline** — `LP1—linea_1—LP2` is one isolated island, so a U-clip at `LP1` alone is a dead
  end. Below, `LP2` is dropped and `linea_1.end` is terminated `'short'`: current enters at the
  `LP1` terminal, runs down the trace, drops to ground at the far end and returns through the
  ground plane. (`ShortToGround` draws no metal of its own — it just lets the route's ground cut
  stop there, so the trace tip touches the plane: a genuine galvanic short.)

### What this solve does not contain

Dropping $\partial\mathbf D/\partial t$ means it knows nothing about capacitance — hence the
separate electrostatic run above for $C$, which is the complementary limit. Together they give
$L'$ and $C'$ per unit length, and the telegrapher's equations rebuild the line from them. Only
a full-wave solve (the driven-modal sweep above) does both at once, with retardation.

### One caveat that does not exist in the capacitance case

The magnetostatic solve forces a **uniform DC current** along the whole meander. The $\lambda/4$
mode does not: with the short at $s=0$, its current profile is $I(s)=I_0\cos(\pi s/2\ell)$. So

$$M_\text{sim}=\int_0^\ell m(s)\,ds \qquad\text{but}\qquad
M_\text{mode}=\int_0^\ell m(s)\cos\!\left(\frac{\pi s}{2\ell}\right)ds$$

For a *localised* coupler at $s_0$ this is just a scale factor,
$M_\text{mode}\approx\cos(\pi s_0/2\ell)\,M_\text{sim}$, with $s_0$ measured from `stg1`. The
feedline needs no such correction: it carries a travelling wave of essentially constant
amplitude over the short coupling stretch.

### Further reading

Ordered by how much it will change the numbers you get, which is *not* the same as ordered by
rigour. Nothing in this notebook went wrong for numerical-analysis reasons; the failures were
modelling ones — a terminal not cutting a closed loop, the wrong quantity extracted, the
standing-wave weighting, the Maxwell-vs-mutual convention. That skill lives in the microwave and
circuit-QED literature, not in the FEM books.

**Worth the time**

* Pozar, *Microwave Engineering*, 4th ed., Wiley. Chapter 6 (*Microwave Resonators*):
  transmission-line resonators, unloaded/loaded/external $Q$, and coupling a resonator to a
  feedline. The chapter that says in two pages that a coupler near the open end is capacitive.
* Göppl et al., *Coplanar waveguide resonators for circuit QED*, J. Appl. Phys. **104**, 113904
  (2008). The paper for this exact object: $L'$ and $C'$ from conformal mapping (what the sanity
  cell above recomputes), lumped equivalents of $\lambda/2$ and $\lambda/4$ lines, capacitive
  coupling to a feedline, $Q_e$.
* Krantz et al., *A Quantum Engineer's Guide to Superconducting Qubits*, Appl. Phys. Rev. **6**,
  021318 (2019). Standard onboarding review.
* Probst et al., Rev. Sci. Instrum. **86**, 024706 (2015). Circle-fit extraction of resonator
  $Q$ — the principled replacement for the 3 dB-width estimator in the driven section, and it
  separates $Q_i$ from $Q_e$.
* Zmuidzinas, *Superconducting Microresonators*, Annu. Rev. Condens. Matter Phys. **3**, 169
  (2012), if the films have appreciable kinetic inductance. PALACE's metals are zero-thickness
  PEC with no London depth, so the $M$ above is the **geometric** part only.

**On the FEM itself, if curious**

* Jin, *The Finite Element Method in Electromagnetics*, 3rd ed., Wiley. The engineer's book:
  edge elements, spurious modes, and the null-space/gauge problem, written for people who
  compute fields rather than prove theorems. If you read one FEM book, this one.
* Hiptmair, *Finite elements in computational electromagnetism*, Acta Numerica (2002). Free,
  rigorous, readable; covers the discrete de Rham complex and the gradient null space.
* Schneebeli, *An H(curl;Ω)-conforming FEM: Nédélec's elements of first type* (2003). Short and
  precise on exactly the edge-element claims made above: §1.2 (tangential continuity is the only
  continuity condition), §2.1.2 (the edge dof *is* $\int_e \mathbf A\cdot d\boldsymbol\ell$, with
  the dof counts per tetrahedron), §2.1.3 (the Piola transform — why componentwise pull-back
  fails), and Appendix B (the low-frequency limit of time-harmonic Maxwell). Note its model
  problem carries a coercive $+c(x)\mathbf u$ term, so Lax–Milgram applies and **the null-space
  problem never appears** — that is the eddy-current case, not the magnetostatic one. Hiptmair
  covers the gap.

**Skip** Monk, *Finite Element Methods for Maxwell's Equations* (Oxford, 2003) and
Boffi–Brezzi–Fortin unless you intend to write a solver. They are numerical-analysis monographs;
they will not change a single number here. The exception: if a solve ever misbehaves in a way
that smells numerical — stagnation with no topological cause, mesh sensitivity, spurious
eigenmodes — Jin's edge-element chapter is the twenty pages that pay off.

In [ ]:
import numpy as np
import shapely
from scipy.special import ellipk

from toolkit.tools.sim import (run_inductance_sim, InductanceSimOptions, CurrentSourceOptions,
                               MeshAlongPathOptions, check_current_loops, plot_current_loops)

In [ ]:
OUTPUT_INDUCTANCE = "./resonator_inductance"
NUM_CPUS_INDUCTANCE = 4

ind_options = InductanceSimOptions(
    solver_order=2,
    solver_tol=1.0e-8,
    solver_maxits=5000,   # AMS on thin PEC sheets with a floating island converges SLOWLY; 200
                          # is nowhere near enough and just hands back a stalled iterate.
    palace_dir=path2palace,
    num_cpus=NUM_CPUS_INDUCTANCE,
)

# --- Terminal 1: the feedline loop --------------------------------------------------------
# LP1's U-clip is the cut. The loop is closed at the OTHER end by dropping LP2 and terminating
# linea_1.end as 'short' (ShortToGround adds no metal of its own -- it just lets the route's
# ground cut stop there, so the trace tip touches the ground plane: a genuine galvanic short).
# Without that short the feedline is an isolated island, the system is inconsistent, and the
# solve diverges.
feedline_source = CurrentSourceOptions(component_name="LP1", type="Uclip_on_Launcher")

# --- Terminal 2: the resonator loop -------------------------------------------------------
# Closed at stg1 already; the cut goes at the free end. The U-clip bridges the two flanking
# ground planes just past the trace tip and the terminal patch spans the gap between them.
resonator_source = CurrentSourceOptions(component_name="meandro_1", type="Uclip_on_Route",
                                        pin_name="end", separation_gap=20e-6)

meander_mesh = MeshAlongPathOptions(qObjName="meandro_1", dist_resolution=10e-6,
                                    min_size=4e-6, max_size=150e-6, taper_dist_min=10e-6)
feedline_mesh = MeshAlongPathOptions(qObjName="linea_1", dist_resolution=10e-6,
                                     min_size=4e-6, max_size=150e-6, taper_dist_min=10e-6)

In [ ]:
# PRE-FLIGHT: does every terminal cut a galvanically closed loop? Two seconds, versus minutes
# for a solve that was never going to work. Take the metal, add the U-clips, remove the terminal
# patches, count the connected pieces -- 1 means every cut has a way around, 2+ means some
# conductor is an island reachable only through its own cut (div J != 0, no solution).
# Same arguments as run_inductance_sim below, so they can be kept in step.
loops = check_current_loops(
    design,
    current_sources=[feedline_source, resonator_source],
    components=['meandro_1', 'stg1', 'linea_1', 'LP1'],
    terminations={('linea_1', 'end'): 'short', ('meandro_1', 'end'): 'open'},
    strict=True,   # raise rather than let a doomed run start
)
print(loops.summary())

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
plot_current_loops(loops, ax=axes[0])                 # whole chip
plot_current_loops(loops, ax=axes[1], zoom=0)         # terminal 1, at LP1
plot_current_loops(loops, ax=axes[2], zoom=1)         # terminal 2, at meandro_1.end
fig.tight_layout()
plt.show()

In [ ]:
# No integration areas: flux through a patch sitting in the gap between the two lines is not a
# cross-check on the mutual. B is tangential on every PEC sheet, so the normal flux through a
# small in-plane patch there is ~0 by construction (the previous run returned 1e-28 Wb). The
# mutual inductance is the flux through the WHOLE resonator loop, which is what terminal-M.csv
# already integrates properly.
inductance_result = run_inductance_sim(
    design=design,
    name="resonator_inductance",
    output_path=OUTPUT_INDUCTANCE,
    components=['meandro_1', 'stg1', 'linea_1', 'LP1'],   # LP2 dropped on purpose, see below
    terminations={('linea_1', 'end'): 'short',            # closes the feedline loop
                  ('meandro_1', 'end'): 'open'},          # keeps the resonator tip off ground so
                                                          # the U-clip terminal has a gap to sit in
    sim_options=ind_options,
    current_sources=[feedline_source, resonator_source],
    fine_mesh_paths=[meander_mesh, feedline_mesh],
    # strict_convergence=True by default: raises if any linear solve stalled, instead of
    # returning a plausible-looking matrix built from a garbage field.
)

In [ ]:
inductance_result

In [ ]:
import pandas as pd

M_mat = inductance_result['inductance_matrix']
names = inductance_result['terminal_names']        # same order the current sources were given in
display(pd.DataFrame(M_mat * 1e12, index=names, columns=names).round(3))   # pH

L_feed, L_res, M_dc = M_mat[0, 0], M_mat[1, 1], M_mat[0, 1]

# Reference value: closed-form CPW inductance per unit length (conformal mapping, zero
# thickness). k = w/(w+2g), L' = (mu0/4) K(k')/K(k); scipy's ellipk takes the PARAMETER m = k^2.
w, g = 20e-6, 11e-6
k2 = (w / (w + 2*g))**2
Lp = np.pi*1e-7 * ellipk(1 - k2) / ellipk(k2)
centreline = lambda c: shapely.LineString(np.array(design.components[c].get_points()) * 1e-3)
ell_feed, ell_res = centreline('linea_1').length, centreline('meandro_1').length

print(f"L feedline  {L_feed*1e9:8.3f} nH   (CPW estimate {Lp*ell_feed*1e9:.2f} nH)")
print(f"L resonator {L_res*1e9:8.3f} nH   (CPW estimate {Lp*ell_res*1e9:.2f} nH)")
print(f"M (DC)      {M_dc*1e12:8.3f} pH   ->  k = {abs(M_dc)/np.sqrt(L_feed*L_res):.2e}")

# Cauchy-Schwarz on the K inner product (see the Inductance markdown): k > 1 does not mean a bad
# mesh, it means what came back is not a field.
assert abs(M_dc) < np.sqrt(L_feed*L_res), "|M| > sqrt(L1 L2): the run is broken, not just inaccurate."
for lab, got, want in (("feedline", L_feed, Lp*ell_feed), ("resonator", L_res, Lp*ell_res)):
    if not 0.4 < got/want < 2.5:
        print(f"  !! {lab} self-inductance is {got/want:.1f}x the CPW estimate -- check the "
              f"terminal topology and the mesh before trusting M.")

### From the DC matrix to the coupling of the mode

`M_dc` is the mutual for a *uniform* current along the resonator. The $\lambda/4$ mode weights
each bit of the coupler by its local current, so what enters the circuit model is
$M_\text{mode}=\bar w_I M_\text{dc}$ with $\bar w_I$ the mean of $\cos(\pi s/2\ell)$ over the
coupler ($s$ from the short). The same geometry also couples capacitively, weighted by the
*voltage* profile $\sin(\pi s/2\ell)$ instead — so a coupler near the shorted end is inductive,
one near the open end capacitive. The cell below compares the two normalised per-length
couplings, $M'/L'$ against $C_m'/C'$, using $C_m$ from the capacitance run. Neither is the whole
answer: the loaded $Q$ from the driven $S_{21}$ sweep includes both channels *and* their
interference.

In [ ]:
# s measured from the short at stg1, which is where meandro_1's centreline starts.
res_ln, feed_ln = centreline('meandro_1'), centreline('linea_1')
assert np.allclose(np.array(design.components['meandro_1'].get_points())[0],
                   design.components['stg1'].pins['short']['middle']), "s measured from the wrong end"

s = np.linspace(0, res_ln.length, 2001)
near = np.array([res_ln.interpolate(x).distance(feed_ln) for x in s]) < 2*(w + 2*g)
w_I = np.cos(np.pi * s[near] / (2*res_ln.length)).mean()    # current  -> inductive channel
w_V = np.sin(np.pi * s[near] / (2*res_ln.length)).mean()    # voltage  -> capacitive channel
ell_c = np.ptp(s[near])

Cp = 4 * 8.8541878128e-12 * (1 + 11.45)/2 * ellipk(k2) / ellipk(1 - k2)   # CPW C', same mapping
ind = w_I * (abs(M_dc) / ell_c) / Lp
cap = w_V * (cupling_cap*1e-15 / ell_c) / Cp

print(f"coupler at s/l = {s[near].mean()/res_ln.length:.3f}, {ell_c*1e6:.0f} um long")
print(f"  M_mode = w_I M_dc = {w_I:.3f} x {M_dc*1e12:.3f} pH = {w_I*M_dc*1e12:.3f} pH")
print(f"  inductive  channel  w_I (M'/L')  = {ind:.2e}")
print(f"  capacitive channel  w_V (Cm'/C') = {cap:.2e}")
print(f"  -> {'INDUCTIVE' if ind > cap else 'CAPACITIVE'}-dominated by {max(ind,cap)/min(ind,cap):.1f}x")